# **Day 4: Pandas for Machine Learning**
*Module 1 - Foundations and Tools*

Real data arrives as **tables** (CSV, Excel, SQL, Parquet) with mixed types, missing values and errors. **Pandas** is the tool for loading, cleaning, joining and summarising tables before they go into a model.

> Pandas handles tables: rows are samples, columns are variables. It can read files, clean them, filter rows, join tables and summarise groups. Most ML projects start as a pandas table.
>
> *Example:* A field-survey spreadsheet with columns plot_id, crop, date and NDVI can be loaded, filtered to one district, and averaged per crop in three lines of pandas.

In [1]:
import numpy as np
import pandas as pd
pd.set_option("display.width", 120)
rng = np.random.default_rng(4)

## **1. Creating a Realistic Dataset**
We simulate a **farm survey**: each row is a field with its district, crop, sowing date, satellite NDVI, rainfall and the yield measured at harvest. Real survey data has typos, missing values and duplicates, so we add those too.

In [2]:
n = 400
df = pd.DataFrame({
    "field_id": [f"F{i:04d}" for i in range(n)],
    "district": rng.choice(["Dhanbad", "Bokaro", "Giridih", "Ranchi"], n, p=[0.3, 0.25, 0.25, 0.2]),
    "crop": rng.choice(["rice", "maize", "wheat"], n, p=[0.5, 0.3, 0.2]),
    "sowing_date": pd.to_datetime("2024-06-01") + pd.to_timedelta(rng.integers(0, 60, n), unit="D"),
    "area_ha": rng.gamma(2.0, 0.6, n).round(2),
    "peak_ndvi": rng.uniform(0.35, 0.9, n).round(3),
    "rain_mm": rng.normal(1100, 200, n).round(0),
})
base = {"rice": 2.0, "maize": 2.6, "wheat": 2.2}
df["yield_t_ha"] = (df.crop.map(base) + 3.5 * df.peak_ndvi + 0.0008 * df.rain_mm
                    - 0.01 * (df.sowing_date - df.sowing_date.min()).dt.days + rng.normal(0, 0.3, n)).round(2)

# add realistic problems
df.loc[rng.choice(n, 20, replace=False), "rain_mm"] = np.nan
df.loc[rng.choice(n, 8, replace=False), "peak_ndvi"] = np.nan
df.loc[rng.choice(n, 5, replace=False), "crop"] = "Rice "          # typo + trailing space
df = pd.concat([df, df.iloc[:6]], ignore_index=True)               # duplicated rows
df.to_csv("farm_survey.csv", index=False)
print("Saved farm_survey.csv with", len(df), "rows")

Saved farm_survey.csv with 406 rows


## **2. Reading and Inspecting**

In [3]:
df = pd.read_csv("farm_survey.csv", parse_dates=["sowing_date"])
df.head()

,field_id,district,crop,sowing_date,area_ha,peak_ndvi,rain_mm,yield_t_ha
0,F0000,Ranchi,rice,2024-07-28,3.02,0.848,872.0,4.84
1,F0001,Bokaro,maize,2024-07-07,1.76,0.624,1304.0,5.12
2,F0002,Ranchi,maize,2024-06-01,0.28,0.855,1123.0,6.47
3,F0003,Dhanbad,maize,2024-07-15,1.64,0.808,891.0,6.25
4,F0004,Giridih,maize,2024-06-28,0.71,0.623,1370.0,5.38


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 406 entries, 0 to 405
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   field_id     406 non-null    str           
 1   district     406 non-null    str           
 2   crop         406 non-null    str           
 3   sowing_date  406 non-null    datetime64[us]
 4   area_ha      406 non-null    float64       
 5   peak_ndvi    398 non-null    float64       
 6   rain_mm      386 non-null    float64       
 7   yield_t_ha   406 non-null    float64       
dtypes: datetime64[us](1), float64(4), str(3)
memory usage: 31.9 KB


In [5]:
df.describe().round(2)

,sowing_date,area_ha,peak_ndvi,rain_mm,yield_t_ha
count,406,406.00,398.00,386.00,406.00
mean,2024-06-30 23:24:31.921182,1.22,0.63,1101.99,5.03
min,2024-06-01 00:00:00,0.04,0.35,389.00,3.09
25%,2024-06-16 00:00:00,0.62,0.50,972.25,4.46
50%,2024-07-01 00:00:00,1.03,0.64,1116.00,5.02
75%,2024-07-16 00:00:00,1.65,0.77,1232.25,5.55
max,2024-07-30 00:00:00,4.72,0.89,1610.00,7.26
std,NaN,0.82,0.16,200.73,0.76


In [6]:
print(df["crop"].value_counts(), "\n")
print("Missing values per column:\n", df.isna().sum())
print("\nDuplicated rows:", df.duplicated().sum())

crop
rice     196
maize    127
wheat     78
Rice       5
Name: count, dtype: int64 

Missing values per column:
 field_id        0
district        0
crop            0
sowing_date     0
area_ha         0
peak_ndvi       8
rain_mm        20
yield_t_ha      0
dtype: int64

Duplicated rows: 6


## **3. Cleaning**
Fix the typo, drop duplicates and look at missing values (Day 17 covers imputation properly).

In [7]:
df["crop"] = df["crop"].str.strip().str.lower()
df = df.drop_duplicates().reset_index(drop=True)
print(df["crop"].value_counts())
print("Rows after removing duplicates:", len(df))

crop
rice     199
maize    123
wheat     78
Name: count, dtype: int64
Rows after removing duplicates: 400


## **4. Selecting Data**

| Method | Selects by | Example |
|---|---|---|
| `df["col"]` / `df[["a","b"]]` | Column names | `df[["peak_ndvi", "rain_mm"]]` |
| `df.loc[rows, cols]` | **Labels** / boolean masks | `df.loc[df.crop == "rice", "yield_t_ha"]` |
| `df.iloc[rows, cols]` | **Positions** | `df.iloc[:5, 2:4]` |
| `df.query("...")` | A readable expression | `df.query("crop == 'rice' and area_ha > 2")` |

In [8]:
print(df.loc[df.crop == "rice", ["district", "yield_t_ha"]].head(3), "\n")
print(df.iloc[:3, 2:5], "\n")
print(df.query("crop == 'maize' and peak_ndvi > 0.8")[["field_id", "peak_ndvi", "yield_t_ha"]].head(3))

  district  yield_t_ha
0   Ranchi        4.84
5   Bokaro        4.33
6   Ranchi        4.69 

    crop sowing_date  area_ha
0   rice  2024-07-28     3.02
1  maize  2024-07-07     1.76
2  maize  2024-06-01     0.28 

   field_id  peak_ndvi  yield_t_ha
2     F0002      0.855        6.47
3     F0003      0.808        6.25
14    F0014      0.860        6.02


## **5. New Columns, Dates and Categories**

In [9]:
df["sowing_doy"] = df.sowing_date.dt.dayofyear                   # day of year
df["sowing_week"] = df.sowing_date.dt.isocalendar().week.astype(int)
df["production_t"] = df.area_ha * df.yield_t_ha
df["ndvi_class"] = pd.cut(df.peak_ndvi, bins=[0, 0.5, 0.7, 1], labels=["low", "medium", "high"])
df["district"] = df["district"].astype("category")
df[["sowing_date", "sowing_doy", "production_t", "ndvi_class"]].head()

,sowing_date,sowing_doy,production_t,ndvi_class
0,2024-07-28,210,14.6168,high
1,2024-07-07,189,9.0112,medium
2,2024-06-01,153,1.8116,high
3,2024-07-15,197,10.2500,high
4,2024-06-28,180,3.8198,medium


## **6. Group, Pivot and Merge**

In [10]:
summary = (df.groupby("crop")
             .agg(n_fields=("field_id", "count"),
                  mean_yield=("yield_t_ha", "mean"),
                  sd_yield=("yield_t_ha", "std"),
                  total_prod=("production_t", "sum"))
             .round(2))
summary

,n_fields,mean_yield,sd_yield,total_prod
crop,,,,
maize,123,5.51,0.72,815.14
rice,199,4.77,0.67,1152.89
wheat,78,4.90,0.67,495.84


In [11]:
pd.pivot_table(df, values="yield_t_ha", index="district", columns="crop", aggfunc="mean", observed=True).round(2)

crop,maize,rice,wheat
district,,,
Bokaro,5.49,4.79,4.86
Dhanbad,5.56,4.61,4.75
Giridih,5.27,4.81,5.01
Ranchi,5.72,4.91,4.93


In [12]:
# Merge with a second table (district-level soil information)
soil = pd.DataFrame({"district": ["Dhanbad", "Bokaro", "Giridih", "Ranchi"],
                     "soil_type": ["laterite", "red", "red", "laterite"],
                     "irrigated_pct": [22, 35, 18, 30]})
df = df.merge(soil, on="district", how="left", validate="many_to_one")
df[["field_id", "district", "soil_type", "irrigated_pct"]].head()

,field_id,district,soil_type,irrigated_pct
0,F0000,Ranchi,laterite,30
1,F0001,Bokaro,red,35
2,F0002,Ranchi,laterite,30
3,F0003,Dhanbad,laterite,22
4,F0004,Giridih,red,18


`validate="many_to_one"` makes pandas raise an error if the soil table had duplicate districts. A silent bad merge can duplicate rows and leak information.

## **7. From DataFrame to X and y**

In [13]:
features = ["peak_ndvi", "rain_mm", "area_ha", "sowing_doy", "irrigated_pct"]
model_df = df.dropna(subset=features + ["yield_t_ha"])
X = model_df[features]
y = model_df["yield_t_ha"]
print("X:", X.shape, "| y:", y.shape)

X_encoded = pd.get_dummies(model_df[features + ["crop"]], columns=["crop"], drop_first=True, dtype=int)
print("With one-hot crop columns:", list(X_encoded.columns))

from sklearn.linear_model import LinearRegression
lr = LinearRegression().fit(X_encoded, y)
print(f"R^2 on training data: {lr.score(X_encoded, y):.3f}")

X: (373, 5) | y: (373,)
With one-hot crop columns: ['peak_ndvi', 'rain_mm', 'area_ha', 'sowing_doy', 'irrigated_pct', 'crop_rice', 'crop_wheat']


R^2 on training data: 0.849


# **Part 2: Going Deeper**
## **Method Chaining, apply vs Vectorisation, Window Functions**

Professional pandas code is written as **chains**. Avoid `apply` with Python functions on large data (it loops in Python); use vectorised column operations.

In [14]:
clean = (pd.read_csv("farm_survey.csv", parse_dates=["sowing_date"])
           .assign(crop=lambda d: d.crop.str.strip().str.lower())
           .drop_duplicates()
           .dropna(subset=["peak_ndvi"])
           .assign(sow_doy=lambda d: d.sowing_date.dt.dayofyear,
                   ndvi_z=lambda d: d.groupby("crop").peak_ndvi.transform(lambda s: (s - s.mean()) / s.std()))
           .sort_values("sowing_date"))
print(clean[["crop", "peak_ndvi", "ndvi_z"]].head())

import time
t = time.perf_counter(); _ = clean.peak_ndvi.apply(lambda v: v * 2 + 1); t1 = time.perf_counter() - t
t = time.perf_counter(); _ = clean.peak_ndvi * 2 + 1; t2 = time.perf_counter() - t
print(f"apply: {t1 * 1e3:.2f} ms | vectorised: {t2 * 1e3:.2f} ms")

      crop  peak_ndvi    ndvi_z
287   rice      0.423 -1.315656
137   rice      0.648  0.126275
2    maize      0.855  1.268991
92    rice      0.615 -0.085208
311   rice      0.837  1.337497
apply: 0.24 ms | vectorised: 0.17 ms


**Group-wise transforms** (`transform`) return a value per row, perfect for features like "NDVI relative to the crop average". **Rolling windows** create time-series features (Day 56):

In [15]:
daily = pd.DataFrame({"date": pd.date_range("2024-06-01", periods=120, freq="D")})
daily["rain"] = rng.gamma(0.6, 12, len(daily)).round(1)
daily = daily.set_index("date")
daily["rain_7d_sum"] = daily.rain.rolling(7).sum()
daily["rain_lag1"] = daily.rain.shift(1)
monthly = daily.rain.resample("MS").sum()
print(daily.head(9)); print("\nMonthly totals:\n", monthly)

            rain  rain_7d_sum  rain_lag1
date                                    
2024-06-01  31.2          NaN        NaN
2024-06-02   5.5          NaN       31.2
2024-06-03   2.7          NaN        5.5
2024-06-04   1.7          NaN        2.7
2024-06-05   2.8          NaN        1.7
2024-06-06   0.1          NaN        2.8
2024-06-07   9.2         53.2        0.1
2024-06-08  15.8         37.8        9.2
2024-06-09   3.2         35.5       15.8

Monthly totals:
 date
2024-06-01    274.3
2024-07-01    244.6
2024-08-01    304.3
2024-09-01    164.7
Freq: MS, Name: rain, dtype: float64


## **Practice Problems**
1. Which district has the highest mean yield for **rice**?
2. Add a column `late_sowing` = True if sowing day-of-year > 180. Compare mean yield of late vs early sowing.
3. Count missing values per district for `rain_mm`.
4. *(Intermediate)* Using `groupby().transform`, fill missing `rain_mm` with the **district median**.

In [16]:
# Solution 1
print(df[df.crop == "rice"].groupby("district", observed=True).yield_t_ha.mean().sort_values(ascending=False).head(1))
# Solution 2
df["late_sowing"] = df.sowing_doy > 180
print(df.groupby("late_sowing").yield_t_ha.mean().round(2))
# Solution 3
print(df.rain_mm.isna().groupby(df.district, observed=True).sum())
# Solution 4
df["rain_filled"] = df.rain_mm.fillna(df.groupby("district", observed=True).rain_mm.transform("median"))
print("Missing after fill:", df.rain_filled.isna().sum())

district
Ranchi    4.905682
Name: yield_t_ha, dtype: float64
late_sowing
False    5.20
True     4.86
Name: yield_t_ha, dtype: float64
district
Bokaro     6
Dhanbad    1
Giridih    5
Ranchi     8
Name: rain_mm, dtype: int64
Missing after fill: 0


## **Key References**
- McKinney, W. (2010). Data structures for statistical computing in Python. *Proceedings of the 9th Python in Science Conference*, 56-61.
- McKinney, W. (2022). *Python for Data Analysis* (3rd ed.). O'Reilly.
- Wickham, H. (2014). Tidy data. *Journal of Statistical Software*, 59(10), 1-23.